# NB13 — ElasticNet & Regularisation Paths

> **Best of both worlds: L1 sparsity + L2 stability for correlated features.**

ElasticNet minimises:
```
SSR + λ₁·Σ|βⱼ| + λ₂·Σβⱼ²
```

Or equivalently (sklearn convention):
```
SSR + α [ (1-l1_ratio)·Σβⱼ²/2 + l1_ratio·Σ|βⱼ| ]
```

- `l1_ratio = 0` → Ridge
- `l1_ratio = 1` → Lasso
- `0 < l1_ratio < 1` → ElasticNet


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression

# Dataset with correlated features — Lasso struggles here
X, y, true_coef = make_regression(n_samples=200, n_features=20,
                                   n_informative=5, noise=10,
                                   coef=True, random_state=0)
X_std = StandardScaler().fit_transform(X)

from sklearn.linear_model import Ridge, Lasso
models = {
    'Ridge (l1=0)':       Ridge(alpha=1).fit(X_std, y),
    'Lasso (l1=1)':       Lasso(alpha=0.1).fit(X_std, y),
    'ElasticNet (l1=0.5)': ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X_std, y),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, m) in zip(axes, models.items()):
    colors = ['steelblue' if c != 0 else 'lightgray' for c in m.coef_]
    ax.bar(range(20), m.coef_, color=colors)
    ax.axhline(0, color='black', linewidth=0.5)
    n_nonzero = np.sum(m.coef_ != 0)
    ax.set_title(f'{name}\n{n_nonzero}/20 non-zero features')
    ax.set_xlabel('Feature index'); ax.grid(alpha=0.3, axis='y')

plt.suptitle('ElasticNet: sparse like Lasso but more stable than Lasso on correlated features')
plt.tight_layout(); plt.show()


In [ ]:
# ElasticNetCV — tune both alpha and l1_ratio simultaneously
from sklearn.linear_model import ElasticNetCV
import numpy as np

en_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    n_alphas=100, cv=5, max_iter=10000
).fit(X_std, y)

print(f"Optimal alpha   : {en_cv.alpha_:.6f}")
print(f"Optimal l1_ratio: {en_cv.l1_ratio_:.2f}")
print(f"Non-zero coefs  : {np.sum(en_cv.coef_ != 0)}/{X.shape[1]}")
print(f"R² (train)      : {en_cv.score(X_std, y):.4f}")


In [ ]:
# Compare all three on the same dataset: train/test R²
from sklearn.model_selection import cross_val_score
import pandas as pd

results = []
for name, m in models.items():
    cv_r2 = cross_val_score(m.__class__(**m.get_params()),
                            X_std, y, cv=5, scoring='r2')
    results.append({'Model': name, 'CV R² mean': cv_r2.mean(), 'CV R² std': cv_r2.std()})

df = pd.DataFrame(results)
print(df.to_string(index=False))


## When to use ElasticNet

- You have many correlated features
- You want some variable selection (Lasso-like) but Ridge's group stability
- In practice: try `l1_ratio ∈ [0.5, 0.9]` and tune via cross-validation

**Next:** NB14 — Cross-validation and model selection.
